# Tutorial Replication

based on this [tutorial](https://medium.com/@maneyogesh065/fine-tuning-biobert-for-custom-named-entity-recognition-a-complete-guide-a05b124edda0)

## generic brain regions

In [5]:
from Bio import Entrez
import numpy as np, torch
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset
from seqeval.metrics import classification_report

Entrez.email = os.environ['email']

brain_regions = [
    "Hippocampus","Amygdala","Thalamus","Hypothalamus","Caudate","Putamen",
    "Globus Pallidus","Substantia Nigra","Cerebellum","Prefrontal Cortex",
    "Motor Cortex","Parietal Cortex","Temporal Cortex","Occipital Cortex",
    "Anterior Cingulate","Posterior Cingulate","Insula",
]
bio_tags = ["O", "B-BRAIN", "I-BRAIN"] 

In [ ]:
def fetch_pubmed_abstracts(query="brain", max_results=100):
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    id_list = Entrez.read(handle)["IdList"]
    abstracts = []
    for pmid in id_list:
        rec = Entrez.read(Entrez.efetch(db="pubmed", id=pmid, retmode="xml"))
        try:
            parts = rec['PubmedArticle'][0]['MedlineCitation']['Article'] \
                       .get('Abstract', {}).get('AbstractText', [])
            text = " ".join(str(p) for p in parts)
            if text.strip():
                abstracts.append(text)
        except Exception:
            pass
    return abstracts

corpus = []
for region in brain_regions:
    corpus += fetch_pubmed_abstracts(region, max_results=200)

In [10]:
import pandas as pd
pd.DataFrame(corpus).to_csv('./fetched_corpus_general.csv')

In [ ]:
import re
phrases = sorted([r.lower().split() for r in brain_regions], key=len, reverse=True)

def label_sentence(words):
    tags = ["O"] * len(words)
    low = [w.lower().strip(".,;:()") for w in words]
    i = 0
    while i < len(words):
        matched = False
        for p in phrases:
            n = len(p)
            if low[i:i+n] == p:
                tags[i] = "B-BRAIN"
                for j in range(1, n):
                    tags[i+j] = "I-BRAIN"
                i += n; matched = True; break
        if not matched:
            i += 1
    return tags

sentences, labels = [], []
for abs in corpus:
    for sent in re.split(r'(?<=[.!?])\s+', abs):
        words = sent.split()
        if not words:
            continue
        tags = label_sentence(words)
        if "B-BRAIN" in tags:
            sentences.append(words)
            labels.append([bio_tags.index(t) for t in tags])

In [12]:
split = int(0.7 * len(sentences))
def make_ds(s, l): return Dataset.from_dict({"tokens": s, "ner_tags": l})
train_ds = make_ds(sentences[:split], labels[:split])
test_ds  = make_ds(sentences[split:], labels[split:])

In [13]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

def tokenize_and_align_labels(examples):
    tok = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True,
                    padding="max_length", max_length=128)
    all_labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tok.word_ids(batch_index=i)
        prev, ids = None, []
        for wid in word_ids:
            if wid is None:
                ids.append(-100)
            elif wid != prev:
                ids.append(label[wid])
            else:
                t = bio_tags[label[wid]]
                ids.append(bio_tags.index("I-BRAIN") if t == "B-BRAIN" else label[wid])
            prev = wid
        all_labels.append(ids)
    tok["labels"] = all_labels
    return tok

train_tok = train_ds.map(tokenize_and_align_labels, batched=True)
test_tok  = test_ds.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/507 [00:00<?, ? examples/s]

Map:   0%|          | 0/218 [00:00<?, ? examples/s]

In [14]:
def tokenize_and_align_labels(examples):
    tok = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True,
                    padding="max_length", max_length=128)
    all_labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tok.word_ids(batch_index=i)
        prev, ids = None, []
        for wid in word_ids:
            if wid is None:
                ids.append(-100)
            elif wid != prev:
                ids.append(label[wid])
            else:
                t = bio_tags[label[wid]]
                ids.append(bio_tags.index("I-BRAIN") if t == "B-BRAIN" else label[wid])
            prev = wid
        all_labels.append(ids)
    tok["labels"] = all_labels
    return tok

train_tok = train_ds.map(tokenize_and_align_labels, batched=True)
test_tok  = test_ds.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/507 [00:00<?, ? examples/s]

Map:   0%|          | 0/218 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "dmis-lab/biobert-v1.1", num_labels=len(bio_tags))

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    tp = [[bio_tags[x] for x, l in zip(pr, la) if l != -100] for pr, la in zip(preds, labels)]
    tl = [[bio_tags[l] for x, l in zip(pr, la) if l != -100] for pr, la in zip(preds, labels)]
    r = classification_report(tl, tp, output_dict=True)["micro avg"]
    return {"precision": r["precision"], "recall": r["recall"], "f1": r["f1-score"]}

args = TrainingArguments(output_dir="../local_experiments/biobert-brain-full", learning_rate=3e-5,
    per_device_train_batch_size=16, num_train_epochs=5, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1")

#Trainer in some versions has tokenizer, otehrwise it has processing_class
trainer = Trainer(model=model, args=args, train_dataset=train_tok,
    eval_dataset=test_tok, processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics)

trainer.train()
print(trainer.evaluate())

### data leakage fix
Keeps the abstracts in the same train/val/test sets

In [ ]:
import random, re
random.seed(42)
abstracts = [a for a in corpus if a.strip()]
random.shuffle(abstracts)
n = len(abstracts)
i1, i2 = int(0.70 * n), int(0.85 * n)
splits = {
    "train": abstracts[:i1],
    "val":   abstracts[i1:i2],
    "test":  abstracts[i2:],
}

In [17]:
def build(abs_list):
    sents, labs = [], []
    for abs in abs_list:
        for sent in re.split(r'(?<=[.!?])\s+', abs):
            words = sent.split()
            if not words:
                continue
            tags = label_sentence(words)
            if "B-BRAIN" in tags:
                sents.append(words)
                labs.append([bio_tags.index(t) for t in tags])
    return Dataset.from_dict({"tokens": sents, "ner_tags": labs}).map(
        tokenize_and_align_labels, batched=True)

train_tok = build(splits["train"])
val_tok   = build(splits["val"])
test_tok  = build(splits["test"])

collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print({k: len(v) for k, v in [("train", train_tok), ("val", val_tok), ("test", test_tok)]})

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/108 [00:00<?, ? examples/s]

{'train': 512, 'val': 105, 'test': 108}


### optuna

In [ ]:
%pip install optuna

In [20]:
def model_init():
    return AutoModelForTokenClassification.from_pretrained(
        "dmis-lab/biobert-v1.1", num_labels=len(bio_tags))

def objective(trial):
    args = TrainingArguments(
        output_dir=f"../local_experiments/optuna/trial_{trial.number}",
        learning_rate=trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        per_device_train_batch_size=trial.suggest_categorical("batch_size", [8, 16, 32]),
        num_train_epochs=trial.suggest_int("epochs", 3, 8),
        weight_decay=trial.suggest_float("weight_decay", 0.0, 0.1),
        warmup_ratio=trial.suggest_float("warmup_ratio", 0.0, 0.2),
        eval_strategy="epoch", save_strategy="no",
        report_to="none", disable_tqdm=True, logging_strategy="no")

    trainer = Trainer(
        model_init=model_init, args=args,
        train_dataset=train_tok, eval_dataset=val_tok,
        processing_class=tokenizer, data_collator=collator,
        compute_metrics=compute_metrics)

    trainer.train()
    return trainer.evaluate()["eval_f1"]

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)

print("best f1:", study.best_value)
print("best params:", study.best_params)

{'eval_loss': '0.07155', 'eval_precision': '0.6218', 'eval_recall': '0.7317', 'eval_f1': '0.6723', 'eval_runtime': '5.821', 'eval_samples_per_second': '18.04', 'eval_steps_per_second': '2.405', 'epoch': '1'}
{'eval_loss': '0.01633', 'eval_precision': '0.8857', 'eval_recall': '0.9451', 'eval_f1': '0.9145', 'eval_runtime': '5.832', 'eval_samples_per_second': '18', 'eval_steps_per_second': '2.4', 'epoch': '2'}
{'eval_loss': '0.013', 'eval_precision': '0.9091', 'eval_recall': '0.9756', 'eval_f1': '0.9412', 'eval_runtime': '5.905', 'eval_samples_per_second': '17.78', 'eval_steps_per_second': '2.371', 'epoch': '3'}
{'train_runtime': '316', 'train_samples_per_second': '4.86', 'train_steps_per_second': '0.608', 'train_loss': '0.1272', 'epoch': '3'}
Trial finished with value: 0.9411764705882352 and parameters: {'learning_rate': 1.827226177606625e-05, 'batch_size': 8, 'epochs': 3, 'weight_decay': 0.015599452033620266, 'warmup_ratio': 0.011616722433639893}. Best is trial 0 with value: 0.9411764705882352.
{'eval_loss': '0.013', 'eval_precision': '0.9091', 'eval_recall': '0.9756', 'eval_f1': '0.9412', 'eval_runtime': '5.463', 'eval_samples_per_second': '19.22', 'eval_steps_per_second': '2.563', 'epoch': '3'}

{'eval_loss': '0.07471', 'eval_precision': '0.5769', 'eval_recall': '0.7317', 'eval_f1': '0.6452', 'eval_runtime': '5.947', 'eval_samples_per_second': '17.66', 'eval_steps_per_second': '2.354', 'epoch': '1'}
{'eval_loss': '0.01801', 'eval_precision': '0.8916', 'eval_recall': '0.9024', 'eval_f1': '0.897', 'eval_runtime': '5.958', 'eval_samples_per_second': '17.62', 'eval_steps_per_second': '2.35', 'epoch': '2'}
{'eval_loss': '0.004185', 'eval_precision': '0.9586', 'eval_recall': '0.9878', 'eval_f1': '0.973', 'eval_runtime': '5.929', 'eval_samples_per_second': '17.71', 'eval_steps_per_second': '2.361', 'epoch': '3'}

### retrain after optuna

In [24]:
from datasets import concatenate_datasets
bp = study.best_params

final_args = TrainingArguments(
    output_dir="../local_experiments/biobert-brain-final",
    learning_rate=bp["learning_rate"],
    per_device_train_batch_size=bp["batch_size"],
    num_train_epochs=bp["epochs"],
    weight_decay=bp["weight_decay"],
    warmup_ratio=bp["warmup_ratio"],
    eval_strategy="no", save_strategy="no",
    report_to="none", disable_tqdm=True)

final_trainer = Trainer(
    model_init=model_init, args=final_args,
    train_dataset=concatenate_datasets([train_tok, val_tok]),
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics)

final_trainer.train()
print("test:", final_trainer.evaluate(test_tok))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'train_runtime': '288', 'train_samples_per_second': '6.426', 'train_steps_per_second': '0.812', 'train_loss': '0.1012', 'epoch': '3'}
{'eval_loss': '0.01196', 'eval_precision': '0.9697', 'eval_recall': '0.9938', 'eval_f1': '0.9816', 'eval_runtime': '5.345', 'eval_samples_per_second': '20.2', 'eval_steps_per_second': '2.619', 'epoch': '3'}
test: {'eval_loss': 0.01195946428924799, 'eval_precision': 0.9696969696969697, 'eval_recall': 0.9937888198757764, 'eval_f1': 0.9815950920245399, 'eval_runtime': 5.3451, 'eval_samples_per_second': 20.205, 'eval_steps_per_second': 2.619, 'epoch': 3.0}


## aal brain regions

In [27]:
import json

with open("../../data/AAL3v1.json", "r", encoding="utf8") as f:
    aal = json.load(f)

In [28]:
alias_to_label = {}

for region in aal.values():
    label = region["aal_name"]
    aliases = set()
    aliases.add(region["full_name"])
    aliases.update(region["synonyms"])

    for alias in aliases:
        alias_to_label[alias.lower()] = label

In [39]:
all_aliases = {}
aal_names = []

for region in aal.values():
    label = region["aal_name"]
    aal_names.append(region['full_name'])
    names = [region["full_name"]] + region["synonyms"]
    for n in names:
        all_aliases[n] = label

In [49]:
phrases = sorted([r.lower().split() for r in aal_names], key=len, reverse=True)

def label_sentence_sub(words):
    tags = ["O"] * len(words)
    low = [w.lower().strip(".,;:()") for w in words]
    i = 0
    while i < len(words):
        matched = False
        for p in phrases:
            n = len(p)
            if low[i:i+n] == p:
                tags[i] = f"B-{p}"
                for j in range(1, n):
                    tags[i+j] = f"I-{p}"
                i += n; matched = True; break
        if not matched:
            i += 1
    return tags

In [50]:
phrases

[['left', 'lobule', 'iv,', 'v', 'of', 'cerebellar', 'hemisphere'],
 ['right', 'lobule', 'iv,', 'v', 'of', 'cerebellar', 'hemisphere'],
 ['left', 'inferior', 'frontal', 'gyrus,', 'opercular', 'part'],
 ['right', 'inferior', 'frontal', 'gyrus,', 'opercular', 'part'],
 ['left', 'inferior', 'frontal', 'gyrus,', 'triangular', 'part'],
 ['right', 'inferior', 'frontal', 'gyrus,', 'triangular', 'part'],
 ['left', 'superior', 'frontal', 'gyrus,', 'medial', 'orbital'],
 ['right', 'superior', 'frontal', 'gyrus,', 'medial', 'orbital'],
 ['left', 'anterior', 'cingulate', '&', 'paracingulate', 'gyri'],
 ['right', 'anterior', 'cingulate', '&', 'paracingulate', 'gyri'],
 ['left', 'middle', 'cingulate', '&', 'paracingulate', 'gyri'],
 ['right', 'middle', 'cingulate', '&', 'paracingulate', 'gyri'],
 ['left', 'calcarine', 'fissure', 'and', 'surrounding', 'cortex'],
 ['right', 'calcarine', 'fissure', 'and', 'surrounding', 'cortex'],
 ['left', 'temporal', 'pole:', 'superior', 'temporal', 'gyrus'],
 ['right

In [ ]:
def build_regional_sub(abs_list):
    sents, labs = [], []
    for abs in abs_list:
        for sent in re.split(r'(?<=[.!?])\s+', abs):
            words = sent.split()
            if not words:
                continue
            tags = label_sentence_sub(words)
            if "B-" in tags:
                sents.append(words)
                labs.append([bio_tags.index(t) for t in tags])
    return Dataset.from_dict({"tokens": sents, "ner_tags": labs}).map(
        tokenize_and_align_labels, batched=True)

train_tok = build(splits["train"])
val_tok   = build(splits["val"])
test_tok  = build(splits["test"])

collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print({k: len(v) for k, v in [("train", train_tok), ("val", val_tok), ("test", test_tok)]})

{'train': 0, 'val': 0, 'test': 0}


In [36]:
full_names

['left precentral gyrus',
 'right precentral gyrus',
 'left superior frontal gyrus, dorsolateral',
 'right superior frontal gyrus, dorsolateral',
 'left middle frontal gyrus',
 'right middle frontal gyrus',
 'left inferior frontal gyrus, opercular part',
 'right inferior frontal gyrus, opercular part',
 'left inferior frontal gyrus, triangular part',
 'right inferior frontal gyrus, triangular part',
 'left ifg pars orbitalis',
 'right ifg pars orbitalis',
 'left rolandic operculum',
 'right rolandic operculum',
 'left supplementary motor area',
 'right supplementary motor area',
 'left olfactory cortex',
 'right olfactory cortex',
 'left superior frontal gyrus, medial',
 'right superior frontal gyrus, medial',
 'left superior frontal gyrus, medial orbital',
 'right superior frontal gyrus, medial orbital',
 'left gyrus rectus',
 'right gyrus rectus',
 'left medial orbital gyrus',
 'right medial orbital gyrus',
 'left anterior orbital gyrus',
 'right anterior orbital gyrus',
 'left poste

brain region mentionsed in the abstracts

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/abstracts.csv")
texts = df["0"].tolist()

In [ ]:
import re

def annotate(text):
    matches = []
    lower = text.lower()
    for alias, label in all_aliases.items():
        pattern = r"\b" + re.escape(alias) + r"\b"
        for m in re.finditer(pattern, lower):
            matches.append({
                "start": m.start(),
                "end": m.end(),
                "text": text[m.start():m.end()],
                "label": label
            })
    return sorted(matches, key=lambda x: x["start"])

In [ ]:
annotate("brain region that works the best is hippocampus or the left precentral gyrus and so on")